[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matth426/Calculus-with-Python-Programming/blob/main/notebooks/calcwp_wk7_higher_order_derivatives.ipynb)

# CALCWP: CALCULUS WITH PYTHON PROGRAMMING
## Week 7 — Higher-Order Derivatives and Implicit Differentiation

**Topics this week:**
- Second, third, and higher-order derivatives
- What higher-order derivatives mean physically (velocity vs. acceleration)
- Nested function calls in Python
- Implicit differentiation — when $y$ isn't isolated on one side

**By the end of this notebook, you should be able to:**
1. Compute second and higher-order derivatives using `sympy.diff()` with an order argument.
2. Interpret the second derivative in a physical context (acceleration).
3. Write Python code that applies a function repeatedly (nested calls).
4. Perform implicit differentiation on an equation where $y$ is not isolated, using `sympy`.

---


## 0. Quick Recap from Week 6

Last week, we learned the power, sum, product, quotient, and chain rules, and used `sympy.diff()` to differentiate exactly. This week, we push that idea further in two directions: first, by differentiating a function **more than once**; second, by differentiating equations where $y$ is tangled up with $x$ instead of neatly isolated as $y = f(x)$.


## 1. Higher-Order Derivatives

The derivative $f'(x)$ is itself a function — so nothing stops us from differentiating it **again**. This gives the **second derivative**, written $f''(x)$ or $\dfrac{d^2y}{dx^2}$. Differentiating again gives the **third derivative** $f'''(x)$, and so on.

**Example:** $ f(x) = x^4 - 3x^2 $

- $ f'(x) = 4x^3 - 6x $ (first derivative)
- $ f''(x) = 12x^2 - 6 $ (second derivative — the derivative *of* $f'(x)$)
- $ f'''(x) = 24x $ (third derivative)


In [ ]:
from sympy import symbols, diff

x = symbols('x')
f = x**4 - 3*x**2

first = diff(f, x)
second = diff(f, x, 2)   # the '2' means: differentiate twice
third = diff(f, x, 3)

print("f'(x)   =", first)
print("f''(x)  =", second)
print("f'''(x) =", third)

Notice the `diff(f, x, 2)` syntax — the third argument tells `sympy` how many times to differentiate. This is equivalent to **nesting** the `diff()` call on itself:

```python
diff(diff(f, x), x)
```

Let's confirm both approaches give the same answer. Since `sympy` might return the two results in different (but mathematically equal) forms, we use `.equals()` instead of `==` — `==` checks whether two expressions are written identically, while `.equals()` checks whether they're mathematically the same, even if they look different.


In [ ]:
# Nested calls: differentiate, then differentiate the result again
second_nested = diff(diff(f, x), x)
print("Using nested diff() calls:", second_nested)
print("Using diff(f, x, 2):      ", second)
print("Do they match?", second_nested.equals(second))

This is a good moment to notice something important about Python: `diff(f, x)` **returns a value** (an expression), and that returned value can immediately be used as the input to another function call — including `diff()` itself. This pattern, called a **nested function call**, will show up constantly in programming, not just in calculus.

**Practice 1.1:** For $ g(x) = x^5 - 4x^3 + 2x $, compute $g'(x)$, $g''(x)$, and $g'''(x)$ using `sympy`. Try it both with the `diff(g, x, n)` syntax and with nested `diff()` calls, and confirm they match.

In [ ]:
# Write your code here


<details>
<summary><b>Click for Solution</b></summary>

```python
g = x**5 - 4*x**3 + 2*x

g1 = diff(g, x, 1)
g2 = diff(g, x, 2)
g3 = diff(g, x, 3)

print("g'(x)   =", g1)
print("g''(x)  =", g2)
print("g'''(x) =", g3)

# Nested version check
print("Nested check (g''):", diff(diff(g, x), x).equals(g2))
```

**Answers:** $g'(x) = 5x^4 - 12x^2 + 2$, $g''(x) = 20x^3 - 24x$, $g'''(x) = 60x^2 - 24$

</details>

## 2. Physical Meaning: Velocity and Acceleration

Recall from Week 5: if $s(t)$ is position, then $s'(t)$ is **velocity** (rate of change of position). What is $s''(t)$?

Since $s''(t)$ is the derivative *of velocity*, it represents the rate of change of velocity — which is **acceleration**.

$$ s(t) \xrightarrow{\text{derivative}} v(t) = s'(t) \xrightarrow{\text{derivative}} a(t) = v'(t) = s''(t) $$

**Example:** A ball's height is $ h(t) = -5t^2 + 20t $ (same function from Week 5's mini-challenge).


In [ ]:
t = symbols('t')
h = -5*t**2 + 20*t

velocity = diff(h, t)
acceleration = diff(h, t, 2)

print("Height:       h(t)  =", h)
print("Velocity:     v(t)  =", velocity)
print("Acceleration: a(t)  =", acceleration)

Notice: the acceleration is a **constant**, $-10$ — this makes sense physically, since gravity pulls the ball down at a constant rate (this constant relates to $g$, the acceleration due to gravity, in the simplified units used here).

**Practice 2.1:** A different object's position is given by $ s(t) = t^3 - 6t^2 + 9t $. Find its velocity and acceleration functions. Then evaluate the velocity at $t=1$ and $t=3$ using `.subs()` — is the object moving in the same direction at both times?

In [ ]:
# Write your code here


<details>
<summary><b>Click for Solution</b></summary>

```python
s = t**3 - 6*t**2 + 9*t

v = diff(s, t)
a = diff(s, t, 2)

print("v(t) =", v)
print("a(t) =", a)

v_at_1 = v.subs(t, 1)
v_at_3 = v.subs(t, 3)
print("v(1) =", v_at_1)
print("v(3) =", v_at_3)
```

**Observation:** $v(1) = 0$ and $v(3) = 0$ — the object is momentarily at rest at both times (these are turning points in its motion, not moving in a consistent direction at either instant).

</details>

## 3. Implicit Differentiation

So far, every function we've differentiated has been **explicit**: $y = f(x)$, with $y$ isolated on one side. But many equations mix $x$ and $y$ together without isolating either — these define $y$ **implicitly**.

**Example:** The equation of a circle with radius 5:

$$ x^2 + y^2 = 25 $$

This is *not* written as $y = f(x)$ — in fact, for a given $x$, there are generally two possible $y$ values (the circle isn't even a function by the vertical line test!). Yet we can still ask: "what is $\dfrac{dy}{dx}$ at a particular point on the circle?"

**The idea:** treat $y$ as an unknown function of $x$ (write it as $y(x)$ in your head), differentiate both sides of the equation with respect to $x$, applying the chain rule whenever you differentiate a $y$-term, then solve algebraically for $\dfrac{dy}{dx}$.

By hand:
$$ \frac{d}{dx}[x^2] + \frac{d}{dx}[y^2] = \frac{d}{dx}[25] $$
$$ 2x + 2y\frac{dy}{dx} = 0 $$
$$ \frac{dy}{dx} = -\frac{x}{y} $$

Let's do this in `sympy`, using `Function` to tell `sympy` that $y$ depends on $x$, and `Eq` to represent the equation.


In [ ]:
from sympy import Function, Eq, solve, Derivative

x = symbols('x')
y = Function('y')(x)   # tells sympy: y is a function of x, not a plain symbol

circle_eq = Eq(x**2 + y**2, 25)

# Differentiate both sides with respect to x
differentiated = Eq(diff(circle_eq.lhs, x), diff(circle_eq.rhs, x))
print("After differentiating both sides:")
print(differentiated)

Notice `sympy` automatically applied the chain rule to $y^2$, producing a term with $\dfrac{dy}{dx}$ (shown as `Derivative(y(x), x)`), exactly like our hand calculation. Now we solve this equation for $\dfrac{dy}{dx}$:

In [ ]:
dy_dx = solve(differentiated, Derivative(y, x))[0]
print("dy/dx =", dy_dx)

This matches our hand-derived result: $ \dfrac{dy}{dx} = -\dfrac{x}{y} $.

We can now evaluate the slope of the circle at a specific point, say $(3, 4)$ (which does satisfy $3^2+4^2=25$):


In [ ]:
slope_at_point = dy_dx.subs({x: 3, y: 4})
print("Slope of the circle at (3, 4):", slope_at_point)

**Practice 3.1:** Use implicit differentiation to find $\dfrac{dy}{dx}$ for the equation $ x^2 + xy + y^2 = 7 $. Then evaluate the slope at the point $(1, 2)$ (check: $1^2 + (1)(2) + 2^2 = 1+2+4=7$ ✓).

In [ ]:
# Write your code here


<details>
<summary><b>Click for Solution</b></summary>

```python
y = Function('y')(x)
eq = Eq(x**2 + x*y + y**2, 7)

differentiated = Eq(diff(eq.lhs, x), diff(eq.rhs, x))
print("After differentiating:", differentiated)

dy_dx = solve(differentiated, Derivative(y, x))[0]
print("dy/dx =", dy_dx)

slope_at_point = dy_dx.subs({x: 1, y: 2})
print("Slope at (1, 2):", slope_at_point)
```

</details>

## 4. Visualizing an Implicit Curve

Since the circle $x^2+y^2=25$ isn't a single function $y=f(x)$, we can't plot it with a simple `plt.plot(x, f(x))` like before. Instead, we can generate the top and bottom halves separately (solving for $y$ in each case) and plot both, along with the tangent line at $(3,4)$ we just computed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_vals = np.linspace(-5, 5, 400)

# Top half: y = sqrt(25 - x^2); Bottom half: y = -sqrt(25 - x^2)
y_top = np.sqrt(25 - x_vals**2)
y_bottom = -np.sqrt(25 - x_vals**2)

plt.plot(x_vals, y_top, color='steelblue')
plt.plot(x_vals, y_bottom, color='steelblue')

# Tangent line at (3, 4) with slope -3/4
px, py = 3, 4
slope = -3/4
tangent_x = np.linspace(0, 6, 10)
tangent_y = py + slope * (tangent_x - px)
plt.plot(tangent_x, tangent_y, '--', color='orange', label=f"tangent at (3,4), slope={slope}")

plt.scatter([px], [py], color='red', zorder=5)
plt.gca().set_aspect('equal')
plt.title("Circle x^2 + y^2 = 25 with Tangent Line at (3, 4)")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()

Notice the tangent line touches the circle at exactly one point, $(3,4)$, and its slope matches what implicit differentiation predicted.

## 5. Mini-Challenge

Consider the curve defined implicitly by:

$$ x^2 y + y^3 = 10 $$

**Tasks:**
1. Use `sympy` (with `Function`, `Eq`, `diff`, and `solve`) to find $\dfrac{dy}{dx}$.
2. Check that the point $(1, 2)$ lies on the curve (i.e., confirm $1^2(2) + 2^3 = 10$).
3. Evaluate the slope of the curve at $(1, 2)$.
4. **Bonus:** Using $f(x) = x^4 - 8x^2$, compute the second derivative with `sympy`, then find where $f''(x) = 0$ using `solve()`. (These points are called **inflection points** — we'll explore them properly in Week 9.)


In [ ]:
# Task 1: Implicit differentiation


# Task 2: Confirm (1, 2) satisfies the equation (as a comment or print statement)


# Task 3: Evaluate the slope at (1, 2)


# Task 4 (Bonus): Second derivative and solve f''(x) = 0


<details>
<summary><b>Click for Solution</b></summary>

```python
# Task 1
y = Function('y')(x)
eq = Eq(x**2 * y + y**3, 10)
differentiated = Eq(diff(eq.lhs, x), diff(eq.rhs, x))
dy_dx = solve(differentiated, Derivative(y, x))[0]
print("dy/dx =", dy_dx)

# Task 2
check = 1**2 * 2 + 2**3
print("Check: 1^2(2) + 2^3 =", check, "-- should equal 10")

# Task 3
slope_at_point = dy_dx.subs({x: 1, y: 2})
print("Slope at (1, 2):", slope_at_point)

# Task 4 (Bonus)
f = x**4 - 8*x**2
f_double_prime = diff(f, x, 2)
print("f''(x) =", f_double_prime)

inflection_candidates = solve(f_double_prime, x)
print("f''(x) = 0 at x =", inflection_candidates)
```

</details>

## Summary

This week, you learned:
- How to compute higher-order derivatives with `diff(f, x, n)`, and that this is equivalent to nesting `diff()` calls
- The physical meaning of the second derivative: acceleration is the derivative of velocity
- How Python's nested function calls work — a function's return value can feed directly into another function call
- Implicit differentiation: differentiating both sides of an equation where $y$ isn't isolated, using `sympy`'s `Function`, `Eq`, and `solve`
- How to evaluate and visualize the slope of an implicitly-defined curve at a specific point

**Next week (Week 8):** We apply everything so far to **rates of change and motion problems**, writing reusable `def` functions that model position, velocity, and acceleration together.

---
*CALCWP — Calculus with Python Programming | Mattheus Marcos Contreras | Lab Manual Series*
